# Chapter 3 — Antenna Fundamentals — equations

Standalone, runnable subset of the master `../RF_Equations.ipynb`, scoped to this chapter.
Run top-to-bottom: **Setup**, then this chapter's sections. All functions are verified against the book's worked examples.

## Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

C = 2.99792458e8        # speed of light, m/s
EPS0 = 8.8541878128e-12 # vacuum permittivity, F/m

def wavelength(f_hz):
    return C / f_hz


## 9. Antenna & Tx Source Model — Ch 3

The transmitter side of the link: how much power radiates in which direction (gain, effective
area, EIRP), where the ray / plane-wave model is valid (far-field), and what polarization
mismatch costs (PLF). Pipeline Tier 0–1 — feeds FSPL / Friis (§1). Triage: §3.2, §3.3, §3.5
are the engine-relevant parts; §3.4 antenna zoo + §3.6 pointing loss are reference/skim.

- Isotropic power density `S = P/(4πd²)` (eq 3.1) = `power_density()` from §0 — the gain reference.
- Gain `G = η·D` (dBi); aperture `G = 4π·Ae/λ²` (eq 3.3), `Ae = η·Ap` (eq 3.2); beamwidth
  rule `G ≈ 26000/(θ_az·θ_el)`. **`Ae = Gλ²/4π` is also the Friis receive aperture** — the
  piece that turns §1's FSPL into a real link.
- **Far-field** `d > 2D²/λ` (eq 3.9): pattern fully formed, gain angle-only, wavefront planar
  ⇒ the **ray-theory validity boundary** (ties to Ch 2). Reactive near-field `r < λ/2π` (eq 3.11).
- Impedance match: `ρ=(Z1−Z0)/(Z1+Z0)`, mismatch loss `1−ρ²`, `VSWR=(1+ρ)/(1−ρ)`.
- Polarization loss `F = cos²τ` (linear, eq for §3.5.2) / full elliptical (eq 3.13); circular↔linear ≈ 3 dB.


In [2]:
def antenna_gain_aperture(Ae_m2, wavelength_m):
    return 4*np.pi*Ae_m2/wavelength_m**2            # eq 3.3 (linear ratio)

def effective_area(gain_linear, wavelength_m):
    return gain_linear*wavelength_m**2/(4*np.pi)    # inverse of 3.3; also the Friis Rx aperture

def effective_area_physical(Ap_m2, eta=0.6):
    return eta*Ap_m2                                # eq 3.2

def gain_from_beamwidth(az_deg, el_deg):
    return 26000.0/(az_deg*el_deg)                  # rule of thumb (linear ratio)

# Example 3.1: 30 cm circular aperture @ 39 GHz
Ap = np.pi*(0.15)**2
Ae = effective_area_physical(Ap, 0.6)
G  = antenna_gain_aperture(Ae, wavelength(39e9))
print(f"Ex 3.1: Ae={Ae:.4f} m^2, G={G:.0f} = {10*np.log10(G):.1f} dBi  (book 39.5 dBi)")
print(f"round-trip effective_area(G): {effective_area(G, wavelength(39e9)):.4f} m^2 (== Ae)")


Ex 3.1: Ae=0.0424 m^2, G=9019 = 39.6 dBi  (book 39.5 dBi)
round-trip effective_area(G): 0.0424 m^2 (== Ae)


In [3]:
def far_field_distance(D_m, wavelength_m):
    return 2*D_m**2/wavelength_m                    # eq 3.9 (Fraunhofer)

def reactive_nearfield_radius(wavelength_m):
    return wavelength_m/(2*np.pi)                   # eq 3.11 (book's convention)
# regions:  reactive NF  r < lambda/2pi  <  radiating NF  <  2D^2/lambda  <  far field

# Example 3.4: 140 MHz monopole
lam3 = wavelength(140e6)
print(f"Ex 3.4: lambda={lam3:.3f} m, reactive near-field r < {reactive_nearfield_radius(lam3):.3f} m (book 0.341 m)")
# a 0.3 m antenna at 2.4 GHz: ray model valid beyond the far-field distance
print(f"0.3 m antenna @2.4 GHz: far-field d > {far_field_distance(0.3, wavelength(2.4e9)):.2f} m")


Ex 3.4: lambda=2.141 m, reactive near-field r < 0.341 m (book 0.341 m)
0.3 m antenna @2.4 GHz: far-field d > 1.44 m


In [4]:
def reflection_coeff_z(Z1, Z0=50.0):
    return abs((Z1 - Z0)/(Z1 + Z0))                 # eq 3.6 (magnitude)

def mismatch_loss(Z1, Z0=50.0):
    return 1 - reflection_coeff_z(Z1, Z0)**2        # eq 3.7 (fraction of power delivered)

def vswr(Z1, Z0=50.0):
    r = reflection_coeff_z(Z1, Z0)
    return (1 + r)/(1 - r)                          # eq 3.8

# Example 3.3: 50-ohm source driving a 73-ohm half-wave dipole
print(f"Ex 3.3: rho={reflection_coeff_z(73):.3f}, VSWR={vswr(73):.2f}, loss={10*np.log10(mismatch_loss(73)):.2f} dB")
print("   (book prints 0.23 / 1.6 / -0.24 dB -- those fit ~80 ohm, not 73; minor slip in the book example)")


Ex 3.3: rho=0.187, VSWR=1.46, loss=-0.15 dB
   (book prints 0.23 / 1.6 / -0.24 dB -- those fit ~80 ohm, not 73; minor slip in the book example)


In [5]:
def plf_linear(tau_deg):
    return np.cos(np.radians(tau_deg))**2           # F = cos^2(tau)  (§3.5.2)

def xpd_linear(tau_deg):
    return np.sin(np.radians(tau_deg))**2           # XPD = sin^2(tau) = 1 - F

def plf_elliptical(AR_w_dB, AR_r_dB, dtau_deg):
    # Full polarization loss factor, eq 3.13. AR in dB; dtau = tilt-angle difference (deg).
    ARw = 10**(AR_w_dB/20.0); ARr = 10**(AR_r_dB/20.0)
    num = ((1+ARw**2)*(1+ARr**2) + 4*ARw*ARr
           + (1-ARw**2)*(1-ARr**2)*np.cos(np.radians(2*dtau_deg)))
    return num/(2*(1+ARw**2)*(1+ARr**2))

print(f"PLF linear: tau=0 -> {plf_linear(0):.2f}, 45 -> {plf_linear(45):.2f}, 90 -> {plf_linear(90):.2f}")
print(f"circular(0 dB) <-> linear(~inf) PLF = {10*np.log10(plf_elliptical(0, 60, 90)):.2f} dB (expect -3)")
# Example 3.5: tx RHCP AR=2 dB, rx AR=3 dB
print(f"Ex 3.5: worst(tau=90)={10*np.log10(plf_elliptical(2,3,90)):.2f} dB, "
      f"best(tau=0)={10*np.log10(plf_elliptical(2,3,0)):.2f} dB (book -0.35 / -0.01)")


PLF linear: tau=0 -> 1.00, 45 -> 0.50, 90 -> 0.00
circular(0 dB) <-> linear(~inf) PLF = -3.00 dB (expect -3)
Ex 3.5: worst(tau=90)=-0.35 dB, best(tau=0)=-0.01 dB (book -0.35 / -0.01)
